# Satellite NO₂ as an Economic Indicator for Myanmar

This notebook presents curated results from exploratory analyses of tropospheric nitrogen dioxide (NO₂) as a proxy for economic activity in Myanmar. All data come from the NASA OMI and ESA TROPOMI satellite instruments.

## Setup & Configuration

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import altair as alt
import altairtheme
import statsmodels.formula.api as smf
from statsmodels.tsa.seasonal import STL
from stargazer.stargazer import Stargazer

altairtheme.enable()
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*condition number.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*tight_layout.*")
from statsmodels.tsa.stattools import grangercausalitytests

In [2]:
def find_project_root(file: str = "pyproject.toml") -> Path:
    """Locate project root by walking up from cwd."""
    current_path = Path().cwd()
    for parent in [current_path, *current_path.parents]:
        if (parent / file).exists():
            return parent
    raise FileNotFoundError(f"{file} not found in any parent directories.")


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data"
print(f"Project root: {PROJECT_ROOT}")


def clean_names(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize column names: strip, lowercase, spaces → underscores."""
    return df.rename(columns=lambda col: col.strip().lower().replace(" ", "_"))


def standardize_series(series: pd.Series) -> pd.Series:
    """Z-score standardize a pandas Series."""
    return (series - series.mean()) / series.std()


def fit_sector_models(
    df: pd.DataFrame, var: str | list[str], outcome: str = "gdp_log"
) -> list[tuple[str, object]]:
    """OLS: outcome ~ var for each economic_activity sector.

    Returns list of (activity_name, fitted_model).
    """
    var_str = " + ".join(var) if isinstance(var, list) else var
    min_obs = 10
    model_list = []
    for activity in sorted(df["economic_activity"].unique()):
        activity_df = df.query("economic_activity == @activity").dropna(
            subset=[outcome] + (var if isinstance(var, list) else [var])
        )
        n = len(activity_df)
        if n < min_obs:
            print(f"  ⚠ Skipping {activity}: only {n} obs (min={min_obs})")
            continue
        print(f"  {activity}: N={n}")
        mod = smf.ols(f"{outcome} ~ {var_str}", data=activity_df).fit()
        model_list.append((activity, mod))
    return model_list


def plot_coefficients(
    model_list: list[tuple[str, object]],
    var: str | list[str] = None,
    title: str = "NO₂ Elasticity of GDP by Sector",
    subtitle: str | None = None,
) -> alt.LayerChart | alt.FacetChart:
    """Plot OLS coefficients with 95% CI error bars from sector models."""
    rows = []
    for activity, mod in model_list:
        params = mod.params.drop("const", errors="ignore")
        conf = mod.conf_int().drop("const", errors="ignore")
        pvals = mod.pvalues.drop("const", errors="ignore")
        for v in params.index:
            if var is not None:
                vars_list = [var] if isinstance(var, str) else list(var)
                if v not in vars_list:
                    continue
            rows.append(
                {
                    "economic_activity": activity,
                    "variable": v,
                    "coefficient": params[v],
                    "std_error": mod.bse[v],
                    "lower": conf.loc[v, 0],
                    "upper": conf.loc[v, 1],
                    "p_value": pvals[v],
                    "significant": pvals[v] < 0.05,
                }
            )

    coef_df = pd.DataFrame(rows)
    if coef_df.empty:
        print("No coefficients to plot.")
        return None

    vars_list = (
        [var]
        if isinstance(var, str)
        else list(var)
        if var is not None
        else coef_df["variable"].unique().tolist()
    )

    base = alt.Chart(coef_df).encode(
        y=alt.Y(
            "economic_activity:N",
            title="",
            sort=alt.EncodingSortField(field="coefficient", order="descending"),
        ),
    )

    points = base.mark_point(filled=True, size=60).encode(
        x=alt.X("coefficient:Q", title="Coefficient"),
        color=alt.condition(
            alt.datum.significant, alt.value("#E3120B"), alt.value("#AAAAAA")
        ),
        tooltip=[
            alt.Tooltip("economic_activity:N", title="Sector"),
            alt.Tooltip("variable:N", title="Variable"),
            alt.Tooltip("coefficient:Q", title="Coefficient", format=".4f"),
            alt.Tooltip("std_error:Q", title="Std. Error", format=".4f"),
            alt.Tooltip("lower:Q", title="Lower CI", format=".4f"),
            alt.Tooltip("upper:Q", title="Upper CI", format=".4f"),
            alt.Tooltip("p_value:Q", title="P-value", format=".4f"),
        ],
    )

    error_bars = base.mark_rule().encode(
        x=alt.X("lower:Q", title=""),
        x2="upper:Q",
        color=alt.condition(
            alt.datum.significant, alt.value("#E3120B"), alt.value("#AAAAAA")
        ),
    )

    zero_line = (
        alt.Chart(pd.DataFrame({"x": [0]}))
        .mark_rule(strokeDash=[4, 4], color="gray")
        .encode(x="x:Q")
    )

    if subtitle is None:
        var_label = " + ".join(vars_list)
        subtitle = f"OLS: gdp_log ~ {var_label} with 95% CI"

    layer = (points + error_bars + zero_line).properties(
        width=500,
        height=300,
        title=alt.Title(text=title, subtitle=subtitle),
    )

    if len(vars_list) > 1:
        return layer.facet(
            facet=alt.Facet("variable:N", title="Variable", sort=vars_list),
            columns=2,
        )
    return layer

Project root: /Users/farhanreynaldo/Documents/world-bank/git-repo/myanmar-economic-monitor


## Load Data

In [3]:
OMI_DIR = DATA_PATH / "AirPollution" / "omi_historic"
CONVERSION_CONSTANT = 6.02214076e19

omi_daily_raw = pd.concat(
    [pd.read_csv(f) for f in sorted(OMI_DIR.glob("omi_no2_myanmar_*.csv"))],
    ignore_index=True,
).query('date < "2026-01-01"')

omi_daily = (
    omi_daily_raw.assign(
        date=lambda df: pd.to_datetime(df["date"]),
        NO2_mean=lambda df: df["NO2_mean"] / CONVERSION_CONSTANT,
    )
    .sort_values("date")
    .reset_index(drop=True)
)

omi_monthly = (
    omi_daily.set_index("date")
    .resample("MS")
    .agg(
        no2=("NO2_mean", "mean"),
        valid_pixels_mean=("valid_pixels", "mean"),
        n_days=("NO2_mean", "count"),
    )
    .assign(log_no2=lambda df: np.log(df["no2"]))
)

In [4]:
gdp_quarterly_raw = pd.read_excel(
    DATA_PATH / "GDP" / "Quarterly GDP per Sector.xlsx",
    sheet_name=1,
    skiprows=1,
    skipfooter=1,
)

gdp_quarterly = (
    gdp_quarterly_raw.pipe(clean_names)
    .melt(
        id_vars=["quarter", "sub_group", "economic_activity"],
        var_name="year",
        value_name="gdp",
    )
    .assign(
        quarter_clean=lambda df: df["quarter"].str.strip(),
        year_first=lambda df: df["year"].str.split("-").str[0].str.strip(),
        year_last=lambda df: df["year"].str.split("-").str[1].str.strip(),
        # Myanmar fiscal year → calendar year:
        # Fiscal Q1 (Apr-Jun) = Cal Q2, Q2 (Jul-Sep) = Q3,
        # Q3 (Oct-Dec) = Q4, Q4 (Jan-Mar) = Q1
        calendar_quarter=lambda df: df["quarter_clean"].map(
            {"Q1": "Q2", "Q2": "Q3", "Q3": "Q4", "Q4": "Q1"}
        ),
        year_selected=lambda df: df.apply(
            lambda row: (
                row["year_last"] if row["quarter_clean"] == "Q4" else row["year_first"]
            ),
            axis=1,
        ),
        date=lambda df: pd.to_datetime(df["year_selected"] + df["calendar_quarter"]),
        sub_group=lambda df: df["sub_group"].str.strip(),
        economic_activity=lambda df: df["economic_activity"].str.strip(),
    )
    .set_index("date")
    .sort_index()
    .groupby(["sub_group", "economic_activity", pd.Grouper(freq="QS")])
    .agg({"gdp": "sum"})
    .reset_index()
    .sort_values(["sub_group", "economic_activity", "date"])
    .assign(
        gdp_log=lambda df: np.log(df["gdp"]),
    )
    # Exclude Agriculture: agricultural NO₂ emissions are negligible
    .query('sub_group != "Agriculture"')
)

<positron-console-cell-4>:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.


In [5]:
ntl_adm0_raw = pd.read_csv(DATA_PATH / "NTL" / "ntl_monthly_adm0_collection2.csv").drop(
    columns=["Unnamed: 0", "Unnamed: 0.1", "geometry"], errors="ignore"
)
ntl_adm1_raw = pd.read_csv(DATA_PATH / "NTL" / "ntl_monthly_adm1_collection2.csv").drop(
    columns=["Unnamed: 0", "Unnamed: 0.1", "geometry"], errors="ignore"
)

ntl_adm0 = (
    ntl_adm0_raw.assign(date=lambda df: pd.to_datetime(df["date"]))
    .pipe(clean_names)
    .set_index("date")
    .groupby(pd.Grouper(freq="QS"))
    .agg({"ntl_mean": "mean", "ntl_sum": "sum"})
    .assign(
        ntl_sum_log=lambda df: np.log(df["ntl_sum"]),
    )
)

ntl_adm1 = (
    ntl_adm1_raw.assign(date=lambda df: pd.to_datetime(df["date"]))
    .pipe(clean_names)
    .set_index("date")
    .rename(columns={"adm1_en": "adm1_name"})
    .groupby(["adm1_name", pd.Grouper(freq="QS")])
    .agg({"ntl_mean": "mean", "ntl_sum": "sum"})
)

In [6]:
ntl_ind_gf_cols = [
    "ntl_ind_5km_sum",
    "ntl_noind_5km_sum",
    "ntl_gf_5km_sum",
    "ntl_nogf_5km_sum",
]

ntl_adm0_sez = pd.read_excel(DATA_PATH / "NTL" / "mmr_adm0_monthly.xlsx").filter(
    ["date", "ADM0_EN", "ADM0_PCODE"] + ntl_ind_gf_cols
)

ntl_adm0_sez_quarterly = (
    ntl_adm0_sez.assign(date=lambda df: pd.to_datetime(df["date"]))
    .set_index("date")
    .groupby(pd.Grouper(freq="QS"))[ntl_ind_gf_cols]
    .sum()
    .assign(
        ntl_ind_5km_sum_log=lambda df: np.log1p(df["ntl_ind_5km_sum"]),
        ntl_gf_5km_sum_log=lambda df: np.log1p(df["ntl_gf_5km_sum"]),
    )
)

In [7]:
era5_monthly = pd.read_csv(
    DATA_PATH
    / "climate"
    / "era5"
    / "myanmar_era5_climate_2019-01-01_2025-12-31_monthly_adm0.csv",
    parse_dates=["date"],
)

met_adm0_quarterly = (
    era5_monthly.set_index("date")
    .resample("QS")
    .agg(
        {
            "region": "first",
            "region_id": "first",
            "precipitation_mm": "sum",
            "temperature_c": "mean",
            "dewpoint_c": "mean",
            "relative_humidity_pct": "mean",
        }
    )
    .reset_index()
)

In [8]:
chirps_monthly = pd.read_csv(
    DATA_PATH
    / "Rainfall"
    / "myanmar_chirps_rainfall_2012-01-01_2025-09-30_monthly_adm0.csv",
    parse_dates=["date"],
)

rainfall_quarterly = (
    chirps_monthly.set_index("date")
    .resample("QS")
    .agg(rainfall_mm=("rainfall_mm", "sum"))
    .assign(rainfall_log=lambda df: np.log(df["rainfall_mm"]))
    .reset_index()
)

> **Fiscal-year note:** Myanmar's fiscal year runs April–March. The GDP data use fiscal quarters (Q1 = Apr–Jun). We map these to calendar quarters for alignment with satellite observations: fiscal Q1 → calendar Q2, etc.

## Quarterly NO₂ Indicators

We aggregate the monthly OMI data to quarterly frequency for alignment with GDP. The quarterly series are constructed by aggregating the daily OMI data to monthly, then to quarterly.

In [9]:
no2_quarterly = (
    omi_monthly.resample("QS")
    .agg(
        no2=("no2", "mean"),
        valid_pixels_mean=("valid_pixels_mean", "mean"),
        n_months=("no2", "count"),
    )
    .assign(
        log_no2=lambda df: np.log(df["no2"]),
        log_no2_lag1=lambda df: np.log(df["no2"]).shift(1),
        log_no2_lag2=lambda df: np.log(df["no2"]).shift(2),
        log_no2_lag3=lambda df: np.log(df["no2"]).shift(3),
    )
    .reset_index()
)

## NO₂ Trend (OMI, 2012–2025)

The OMI instrument provides a 13-year monthly record of tropospheric NO₂ over Myanmar. There's a noticeable decline in NO₂ levels starting in early 2021, following the coup.

In [10]:
nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date"], empty=False
)

base = alt.Chart(omi_monthly.reset_index()).encode(
    x=alt.X("date:T", title="Date"),
)

monthly_line = base.mark_line().encode(
    y=alt.Y("no2:Q", title="NO₂ (mol/m²)", scale=alt.Scale(zero=False)),
)

rules = (
    base.mark_rule(color="gray")
    .encode(
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Month", format="%Y-%m"),
            alt.Tooltip("no2:Q", title="NO₂", format=".3e"),
        ],
    )
    .add_params(nearest)
)

chart = (
    (monthly_line + rules)
    .properties(width=600, height=300)
    .properties(
        title=alt.Title(
            text="Myanmar National NO₂ (OMI, 2012–2025)",
            subtitle="Monthly mean tropospheric column density",
        )
    )
)
chart

alt.LayerChart(...)

The figure below shows the OMI NO₂ time series decomposed into trend, seasonal, and residual components using STL decomposition. The trend component confirms a structural decline in NO₂ beginning around the coup. Seasonal pattern is roughly constant across the 13-year period, consistent with metereological drivers of NO₂.

In [11]:
stl = STL(omi_monthly["no2"], period=12, robust=True)
stl_result = stl.fit()

decomp_df = pd.DataFrame(
    {
        "Observed": stl_result.observed,
        "Trend": stl_result.trend,
        "Seasonal": stl_result.seasonal,
        "Residual": stl_result.resid,
    }
).reset_index()

decomp_long = decomp_df.melt(id_vars=["date"], var_name="Component", value_name="value")

nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date", "Component"], empty=False
)

base = alt.Chart(decomp_long)

lines = base.mark_line().encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("value:Q", title=""),
    color=alt.Color("Component:N", legend=None),
)

rules = (
    base.mark_rule(color="gray")
    .encode(
        x="date:T",
        opacity=alt.when(nearest).then(alt.value(0.2)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-%m"),
            alt.Tooltip("Component:N", title="Component"),
            alt.Tooltip("value:Q", title="Value", format=".3e"),
        ],
    )
    .add_params(nearest)
)

stl_chart = (
    alt.layer(lines, rules)
    .properties(width=650, height=110)
    .facet(
        facet=alt.Facet(
            "Component:N",
            title=None,
            sort=["Observed", "Trend", "Seasonal", "Residual"],
        ),
        columns=1,
    )
    .properties(
        title=alt.Title(
            text="STL Decomposition of National NO₂ (Monthly, OMI)",
            subtitle="Period = 12 months, robust fitting",
        ),
    )
    .resolve_scale(y="independent")
)
stl_chart

alt.FacetChart(...)

## Seasonal Pattern

NO₂ in Myanmar exhibits strong seasonality, where concentrations peak in the dry season (Oct–Mar) when precipitation-driven washout is minimal, while the monsoon (Jun–Sep) suppresses NO₂ levels through wet deposition.

In [12]:
seasonal_df = omi_monthly.reset_index().assign(
    month=lambda df: df["date"].dt.month,
    month_name=lambda df: df["date"].dt.strftime("%b"),
    year=lambda df: df["date"].dt.year,
)

boxplot = (
    alt.Chart(seasonal_df)
    .mark_boxplot(extent="min-max")
    .encode(
        x=alt.X(
            "month:O",
            title="Month",
            axis=alt.Axis(
                labelExpr="['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][datum.value-1]"
            ),
        ),
        y=alt.Y("no2:Q", title="NO₂ (mol/m²)", scale=alt.Scale(zero=False)),
        tooltip=[alt.Tooltip("no2:Q", format=".3e")],
    )
    .properties(
        width=600,
        height=300,
        title=alt.Title(
            text="Seasonal Distribution of Monthly NO₂ (2012–2025)",
            subtitle="Boxplot across all years by calendar month. Dry season (Oct–Mar) typically higher.",
        ),
    )
)
boxplot

alt.Chart(...)

## GDP vs NO₂ Trends by Sector

The following figures show the standardized (z-scored) quarterly OMI NO₂ alongside GDP across all sectors. NO₂ and GDP move broadly together for Manufacturing and Construction, which are the two most combustion-intensive sectors. Mining and Trade show weaker visual co-movement but similar seasonal patterns.

In [13]:
gdp_no2 = (
    gdp_quarterly.merge(no2_quarterly, on="date", how="left")
    .merge(rainfall_quarterly, on="date", how="left")
    .assign(quarter=lambda df: df["date"].dt.quarter.astype(str))
    .dropna(subset=["no2", "gdp"])
)

In [14]:
gdp_no2_standardized_df = (
    gdp_no2.assign(
        gdp_standardized=lambda df: df.groupby("economic_activity")["gdp"].transform(
            standardize_series
        ),
        no2_standardized=lambda df: df.groupby("economic_activity")["no2"].transform(
            standardize_series
        ),
    )
    .melt(
        id_vars=["date", "economic_activity"],
        value_vars=["gdp_standardized", "no2_standardized"],
    )
    .assign(
        variable=lambda df: df["variable"].replace(
            {
                "gdp_standardized": "GDP (Standardized)",
                "no2_standardized": "NO₂ (Standardized)",
            }
        ),
    )
)

nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date", "economic_activity"], empty=False
)

base = alt.Chart(gdp_no2_standardized_df)

lines = base.mark_line().encode(
    x=alt.X("date:T", title="", axis=alt.Axis(format="%Y-Q%q")),
    y=alt.Y("value:Q", title="Standardized Value", axis=alt.Axis(format=".2f")),
    color=alt.Color(
        "variable:N",
        title="Variable",
    ),
)

rules = (
    base.transform_pivot(
        "variable", value="value", groupby=["date", "economic_activity"]
    )
    .mark_rule(color="gray")
    .encode(
        x=alt.X("date:T"),
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-Q%q"),
            alt.Tooltip("GDP (Standardized):Q", title="GDP", format=".2f"),
            alt.Tooltip("NO₂ (Standardized):Q", title="NO₂", format=".2f"),
        ],
    )
    .add_params(nearest)
)

chart = (
    alt.layer(lines, rules)
    .properties(width=300, height=150)
    .facet(
        facet=alt.Facet("economic_activity:N", title=None),
        columns=3,
    )
    .properties(
        title=alt.Title(
            text="Standardized GDP and NO₂ by Economic Activity",
            subtitle="Comparing standardized values (z-scores) of GDP and NO₂ levels across different economic sectors",
        )
    )
    .resolve_scale(y="independent")
)

chart

alt.FacetChart(...)

In [15]:
SELECTED_SECTORS = [
    "Manufacturing",
    "Construction",
    "Mining",
    "Trade",
    "Social and Administrative",
]

nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date", "economic_activity"], empty=False
)

base = alt.Chart(
    gdp_no2_standardized_df.query("economic_activity in @SELECTED_SECTORS")
)

lines = base.mark_line().encode(
    x=alt.X("date:T", title="", axis=alt.Axis(format="%Y-Q%q")),
    y=alt.Y("value:Q", title="Standardized Value", axis=alt.Axis(format=".2f")),
    color=alt.Color("variable:N", title="Variable"),
)

rules = (
    base.transform_pivot(
        "variable", value="value", groupby=["date", "economic_activity"]
    )
    .mark_rule(color="gray")
    .encode(
        x=alt.X("date:T"),
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-Q%q"),
            alt.Tooltip("GDP (Standardized):Q", title="GDP", format=".2f"),
            alt.Tooltip("NO₂ (Standardized):Q", title="NO₂", format=".2f"),
        ],
    )
    .add_params(nearest)
)

chart = (
    alt.layer(lines, rules)
    .properties(width=300, height=180)
    .facet(
        facet=alt.Facet("economic_activity:N", title=None),
        columns=3,
    )
    .properties(
        title=alt.Title(
            text="Standardized GDP and NO₂ by Sector",
            subtitle=[
                "Comparing standardized values (z-scores) of GDP and NO₂ levels across subset of economic sectors (Manufacturing, ",
                "Construction, Mining, Trade, Social and Administrative)",
            ],
        )
    )
    .resolve_scale(y="independent")
)
chart

alt.FacetChart(...)

The following figures show quarter-over-quarter and year-over-year percentage changes in OMI NO₂ and GDP. The co-movement is less visually apparent in the growth rates, but there are still some periods of alignment.

In [16]:
gdp_no2_pct_change_df = (
    gdp_no2.assign(
        gdp_pct_change=lambda df: df.groupby("economic_activity")["gdp"].pct_change(),
        no2_pct_change=lambda df: df.groupby("economic_activity")["no2"].pct_change(),
    )
    .melt(
        id_vars=["date", "economic_activity"],
        value_vars=["gdp_pct_change", "no2_pct_change"],
    )
    .assign(
        variable=lambda df: df["variable"].replace(
            {
                "gdp_pct_change": "GDP Percent Change",
                "no2_pct_change": "NO₂ Percent Change",
            }
        ),
    )
)

nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date", "economic_activity"], empty=False
)

base = alt.Chart(gdp_no2_pct_change_df)

lines = base.mark_line().encode(
    x=alt.X("date:T", title="", axis=alt.Axis(format="%Y-Q%q")),
    y=alt.Y("value:Q", title="", axis=alt.Axis(format=".1%")),
    color=alt.Color(
        "variable:N",
        title="Variable",
    ),
)

rules = (
    base.transform_pivot(
        "variable", value="value", groupby=["date", "economic_activity"]
    )
    .mark_rule(color="gray")
    .encode(
        x=alt.X("date:T"),
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-Q%q"),
            alt.Tooltip("GDP Percent Change:Q", title="GDP", format=".2%"),
            alt.Tooltip("NO₂ Percent Change:Q", title="NO₂", format=".2%"),
        ],
    )
    .add_params(nearest)
)

chart = (
    alt.layer(lines, rules)
    .properties(width=300, height=150)
    .facet(
        facet=alt.Facet("economic_activity:N", title=None),
        columns=3,
    )
    .properties(
        title=alt.Title(
            text="GDP and NO₂ Percent Change (QoQ) by Economic Activity",
            subtitle="Comparing quarter-over-quarter percent change of GDP and NO₂ across economic sectors",
        )
    )
    .resolve_scale(y="independent")
)

chart

alt.FacetChart(...)

In [17]:
gdp_no2_yoy_change_df = (
    gdp_no2.assign(
        gdp_pct_change=lambda df: df.groupby("economic_activity")["gdp"].pct_change(
            periods=4
        ),
        no2_pct_change=lambda df: df.groupby("economic_activity")["no2"].pct_change(
            periods=4
        ),
    )
    .melt(
        id_vars=["date", "economic_activity"],
        value_vars=["gdp_pct_change", "no2_pct_change"],
    )
    .assign(
        variable=lambda df: df["variable"].replace(
            {
                "gdp_pct_change": "GDP Percent Change",
                "no2_pct_change": "NO₂ Percent Change",
            }
        ),
    )
)

nearest = alt.selection_point(
    nearest=True, on="pointerover", fields=["date", "economic_activity"], empty=False
)

base = alt.Chart(gdp_no2_yoy_change_df)

lines = base.mark_line().encode(
    x=alt.X("date:T", title="", axis=alt.Axis(format="%Y-Q%q")),
    y=alt.Y("value:Q", title="", axis=alt.Axis(format=".1%")),
    color=alt.Color(
        "variable:N",
        title="Variable",
    ),
)

rules = (
    base.transform_pivot(
        "variable", value="value", groupby=["date", "economic_activity"]
    )
    .mark_rule(color="gray")
    .encode(
        x=alt.X("date:T"),
        opacity=alt.when(nearest).then(alt.value(0.3)).otherwise(alt.value(0)),
        tooltip=[
            alt.Tooltip("date:T", title="Date", format="%Y-Q%q"),
            alt.Tooltip("GDP Percent Change:Q", title="GDP", format=".2%"),
            alt.Tooltip("NO₂ Percent Change:Q", title="NO₂", format=".2%"),
        ],
    )
    .add_params(nearest)
)

chart = (
    alt.layer(lines, rules)
    .properties(width=300, height=150)
    .facet(
        facet=alt.Facet("economic_activity:N", title=None),
        columns=3,
    )
    .properties(
        title=alt.Title(
            text="GDP and NO₂ Percent Change (YoY) by Economic Activity",
            subtitle="Comparing year-over-year percent change of GDP and NO₂ across economic sectors",
        )
    )
    .resolve_scale(y="independent")
)

chart

alt.FacetChart(...)

## Rainfall and NO₂

Monsoon rainfall physically removes NO₂ from the atmosphere through wet deposition. The charts below show how CHIRPS rainfall and OMI NO₂ move in opposite directions seasonally

In [18]:
no2_rain_monthly = (
    omi_monthly.reset_index()
    .merge(chirps_monthly, on="date", how="inner")
    .assign(
        no2_std=lambda df: standardize_series(df["no2"]),
        rainfall_std=lambda df: standardize_series(df["rainfall_mm"]),
    )
)

no2_rain_long = no2_rain_monthly.melt(
    id_vars=["date"],
    value_vars=["no2_std", "rainfall_std"],
    var_name="variable",
    value_name="standardized_value",
).assign(
    variable=lambda df: df["variable"].map(
        {"no2_std": "NO₂ (OMI)", "rainfall_std": "Rainfall (CHIRPS)"}
    )
)

alt.Chart(no2_rain_long).mark_line(opacity=0.8).encode(
    x=alt.X("date:T", title=""),
    y=alt.Y("standardized_value:Q", title="Standardized Value (z-score)"),
    color=alt.Color("variable:N", title=""),
).properties(
    width=700,
    height=300,
    title=alt.Title(
        text="Monthly NO₂ vs Rainfall (Standardized)",
        subtitle="The rainfall supresses NO₂ by washing out pollutants",
    ),
)

alt.Chart(...)

In [19]:
rainfall_seasonal_df = chirps_monthly.assign(
    month=lambda df: df["date"].dt.month,
)

alt.Chart(rainfall_seasonal_df).mark_boxplot(extent="min-max").encode(
    x=alt.X(
        "month:O",
        title="Month",
        axis=alt.Axis(
            labelExpr="['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][datum.value-1]"
        ),
    ),
    y=alt.Y("rainfall_mm:Q", title="Rainfall (mm)", scale=alt.Scale(zero=False)),
    tooltip=[alt.Tooltip("rainfall_mm:Q", format=".1f")],
).properties(
    width=600,
    height=300,
    title=alt.Title(
        text="Seasonal Distribution of Monthly Rainfall (CHIRPS, 2012–2025)",
        subtitle=[
            "Rainfall peaks during Jun–Sep monsoon season, which coincides with the NO₂ decline",
            "due to washout effects.",
        ],
    ),
)

alt.Chart(...)

## Data Preparation for Regressions

In [20]:
indicators = (
    gdp_quarterly.merge(no2_quarterly, on="date", how="left")
    .merge(ntl_adm0, on="date", how="left")
    .merge(ntl_adm0_sez_quarterly, on="date", how="left")
    .assign(quarter=lambda df: df["date"].dt.quarter.astype(str))
    .dropna(subset=["no2", "gdp", "ntl_sum"])
)

In [21]:
no2_indicators_yoy = (
    gdp_no2.copy()
    .sort_values(["economic_activity", "date"])
    .assign(
        gdp_yoy=lambda df: df.groupby("economic_activity")["gdp"].pct_change(periods=4),
        no2_yoy=lambda df: df.groupby("economic_activity")["no2"].pct_change(periods=4),
        no2_yoy_lag1=lambda df: df.groupby("economic_activity")["no2_yoy"].shift(1),
        no2_yoy_lag2=lambda df: df.groupby("economic_activity")["no2_yoy"].shift(2),
        no2_yoy_lag3=lambda df: df.groupby("economic_activity")["no2_yoy"].shift(3),
    )
)

In [22]:
no2_indicators_qoq = (
    gdp_no2.copy()
    .sort_values(["economic_activity", "date"])
    .assign(
        gdp_qoq=lambda df: df.groupby("economic_activity")["gdp"].pct_change(periods=1),
        no2_qoq=lambda df: df.groupby("economic_activity")["no2"].pct_change(periods=1),
        no2_qoq_lag1=lambda df: df.groupby("economic_activity")["no2_qoq"].shift(1),
        no2_qoq_lag2=lambda df: df.groupby("economic_activity")["no2_qoq"].shift(2),
        no2_qoq_lag3=lambda df: df.groupby("economic_activity")["no2_qoq"].shift(3),
    )
)

In [23]:
omi_adm1_2010 = pd.read_csv(
    DATA_PATH / "AirPollution" / "omi_no2_monthly_adm1_2010_2020.csv"
)
omi_adm1_2021 = pd.read_csv(
    DATA_PATH / "AirPollution" / "omi_no2_monthly_adm1_2021_2025.csv"
)
omi_adm1_raw = pd.concat([omi_adm1_2010, omi_adm1_2021], ignore_index=True)

# Convert from molecules/cm² to mol/m²
omi_adm1_monthly = (
    omi_adm1_raw.assign(
        date=lambda df: pd.to_datetime(df["date"]),
        adm1_name=lambda df: df["ST"].str.strip(),
        no2=lambda df: df["no2_trop_mean"] / CONVERSION_CONSTANT,
    )
    .filter(["date", "adm1_name", "no2", "pixel_count"])
    .assign(no2=lambda df: df["no2"].where(df["no2"] > 0, np.nan))
)

print(
    f"OMI ADM1 monthly: {len(omi_adm1_monthly)} rows, "
    f"{omi_adm1_monthly['adm1_name'].nunique()} regions, "
    f"{omi_adm1_monthly['date'].min():%Y-%m} to {omi_adm1_monthly['date'].max():%Y-%m}"
)

OMI ADM1 monthly: 3420 rows, 18 regions, 2010-01 to 2025-10


## NO₂–GDP Regressions

We estimate sector-by-sector log-log OLS regressions, first without and then with quarter dummies.

In [24]:
print("Model 1: gdp_log ~ log_no2 (all sectors including Energy)")
model_list_l0 = fit_sector_models(gdp_no2, "log_no2")

star_l0 = Stargazer([m for _, m in model_list_l0])
star_l0.custom_columns([a for a, _ in model_list_l0])
display(star_l0)

display(
    plot_coefficients(
        model_list_l0,
        "log_no2",
        title="National Quarterly NO₂–GDP Elasticity (No Lag) (Model 1)",
        subtitle="Seasonality is not accounted for",
    )
)

Model 1: gdp_log ~ log_no2 (all sectors including Energy)
  Communication: N=53
  Construction: N=53
  Electricity: N=53
  Energy: N=53
  Financial Insitutions: N=53
  Manufacturing: N=53
  Mining: N=53
  Rental and other services: N=53
  Social and Administrative: N=53
  Trade: N=53
  Transportation: N=53


alt.LayerChart(...)

**Interpretation:** Manufacturing and Construction show significant positive correlation with NO₂. We expected Transportation and Trade to also show positive correlation, but that is not the case. Energy GDP values seem erratic, thus we exclude Energy sector from all subsequent models.

### Model 2: 3-Quarter Lag

In [25]:
print("Model 2: gdp_log ~ log_no2_lag3 (excluding Energy)")
gdp_no2_no_energy = gdp_no2.query('economic_activity != "Energy"')
model_list_l3 = fit_sector_models(gdp_no2_no_energy, "log_no2_lag3")

star_l3 = Stargazer([m for _, m in model_list_l3])
star_l3.custom_columns([a for a, _ in model_list_l3])
display(star_l3)

display(
    plot_coefficients(
        model_list_l3,
        "log_no2_lag3",
        title="National Quarterly NO₂–GDP Elasticity (3-Quarter Lag) (Model 2)",
        subtitle="Seasonality is not accounted for",
    )
)

Model 2: gdp_log ~ log_no2_lag3 (excluding Energy)
  Communication: N=50
  Construction: N=50
  Electricity: N=50
  Financial Insitutions: N=50
  Manufacturing: N=50
  Mining: N=50
  Rental and other services: N=50
  Social and Administrative: N=50
  Trade: N=50
  Transportation: N=50


alt.LayerChart(...)

**Interpretation:** All sectors where we expect a positive NO₂–GDP correlation (Transportation, Manufacturing, Construction, Trade) show positive coefficients with the 3-quarter lag. We are currently unable to fully explain this lag structure. Is there evidence from Myanmar's National Accounts methodology or reporting delays that could support a 3-quarter lag?

In [26]:
granger_sectors = [
    "Transportation",
    "Manufacturing",
    "Construction",
    "Trade",
    "Social and Administrative",
]
print("Granger Causality Tests (NO₂ → GDP):")
print("=" * 60)
for sector in granger_sectors:
    sector_df = (
        gdp_no2_no_energy.query("economic_activity == @sector")
        .dropna(subset=["gdp_log", "log_no2"])
        .sort_values("date")
    )
    print(f"\n{sector} (N={len(sector_df)}):")
    try:
        result = grangercausalitytests(
            sector_df[["gdp_log", "log_no2"]].values, maxlag=4, verbose=True
        )
    except Exception as e:
        print(f"  Error: {e}")

Granger Causality Tests (NO₂ → GDP):

Transportation (N=53):

Granger Causality
number of lags (no zero) 1
ssr based F test:         F=49.9239 , p=0.0000  , df_denom=49, df_num=1
ssr based chi2 test:   chi2=52.9805 , p=0.0000  , df=1
likelihood ratio test: chi2=36.5316 , p=0.0000  , df=1
parameter F test:         F=49.9239 , p=0.0000  , df_denom=49, df_num=1

Granger Causality
number of lags (no zero) 2
ssr based F test:         F=5.0036  , p=0.0108  , df_denom=46, df_num=2
ssr based chi2 test:   chi2=11.0950 , p=0.0039  , df=2
likelihood ratio test: chi2=10.0388 , p=0.0066  , df=2
parameter F test:         F=5.0036  , p=0.0108  , df_denom=46, df_num=2

Granger Causality
number of lags (no zero) 3
ssr based F test:         F=6.1501  , p=0.0014  , df_denom=43, df_num=3
ssr based chi2 test:   chi2=21.4538 , p=0.0001  , df=3
likelihood ratio test: chi2=17.8514 , p=0.0005  , df=3
parameter F test:         F=6.1501  , p=0.0014  , df_denom=43, df_num=3

Granger Causality
number of lags (no z

The granger causality tests shows a very strong, statistically significant "causal" link from NO₂ levels to GDP across all four sectors.

### Model 3: NO₂ + Overall NTL

In [27]:
print(
    "Model 3: gdp_log ~ log_no2 + ntl_sum_log (excluding Energy, no seasonal controls)"
)
indicators_no_energy = indicators.query('economic_activity != "Energy"')
model_list_ntl = fit_sector_models(indicators_no_energy, ["log_no2", "ntl_sum_log"])

star_ntl = Stargazer([m for _, m in model_list_ntl])
star_ntl.custom_columns([a for a, _ in model_list_ntl])
display(star_ntl)

display(
    plot_coefficients(
        model_list_ntl,
        ["log_no2", "ntl_sum_log"],
        title="NO₂ and Nighttime Lights Elasticity of GDP (Model 3)",
        subtitle="Seasonality is not accounted for, but includes nighttime lights as a control variable",
    )
)

Model 3: gdp_log ~ log_no2 + ntl_sum_log (excluding Energy, no seasonal controls)
  Communication: N=53
  Construction: N=53
  Electricity: N=53
  Financial Insitutions: N=53
  Manufacturing: N=53
  Mining: N=53
  Rental and other services: N=53
  Social and Administrative: N=53
  Trade: N=53
  Transportation: N=53


alt.FacetChart(...)

**Interpretation:** Manufacturing NTL coefficient is negative, suggesting that national-level total NTL is not a good proxy for manufacturing activity at this aggregation level. We need to go one level down (SEZ level) to isolate manufacturing activity. Use the proportion from ADM1 level GDP for Yangon and Mandalay for running the regression.

### Model 4: Seasonal Controls

In [28]:
print("Model 4: gdp_log ~ log_no2 + quarter (excluding Energy)")
model_list_l0q = fit_sector_models(gdp_no2_no_energy, ["log_no2", "quarter"])

star_l0q = Stargazer([m for _, m in model_list_l0q])
star_l0q.custom_columns([a for a, _ in model_list_l0q])
display(star_l0q)

display(
    plot_coefficients(
        model_list_l0q,
        "log_no2",
        title="NO₂ Elasticity of GDP (Seasonal Controls) (Model 4)",
        subtitle="Includes quarter fixed effects to control for seasonality",
    )
)

Model 4: gdp_log ~ log_no2 + quarter (excluding Energy)
  Communication: N=53
  Construction: N=53
  Electricity: N=53
  Financial Insitutions: N=53
  Manufacturing: N=53
  Mining: N=53
  Rental and other services: N=53
  Social and Administrative: N=53
  Trade: N=53
  Transportation: N=53


alt.LayerChart(...)

### Model 4a: Rainfall Control (No Quarter FE)

We control for the physical mechanism behind NO₂ seasonality by adding rainfall as a control variable instead of quarter fixed effects.

In [29]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

met_controls_vif = ["rainfall_log"]

# Full model: NO2 + met controls
full_vars = ["log_no2"] + met_controls_vif
full_vif_df = gdp_no2_no_energy[full_vars].dropna()
full_vif_results = pd.DataFrame(
    {
        "Variable": full_vars,
        "VIF": [
            variance_inflation_factor(full_vif_df.values, i)
            for i in range(len(full_vars))
        ],
    }
)
display(
    full_vif_results.style.format({"VIF": "{:.1f}"}).set_caption(
        "VIF — NO₂ + Meteorological Controls"
    )
)

,Variable,VIF
0,log_no2,35.6
1,rainfall_log,35.6


In [30]:
print("Model 4a: gdp_log ~ log_no2 + rainfall_log (excluding Energy)")
model_list_rain = fit_sector_models(gdp_no2_no_energy, ["log_no2", "rainfall_log"])

star_rain = Stargazer([m for _, m in model_list_rain])
star_rain.custom_columns([a for a, _ in model_list_rain])
display(star_rain)

display(
    plot_coefficients(
        model_list_rain,
        "log_no2",
        title="Model 4a: NO₂ Elasticity of GDP (Rainfall Control)",
        subtitle="OLS: gdp_log ~ log_no2 + rainfall_log | OMI 2012–2025",
    )
)

Model 4a: gdp_log ~ log_no2 + rainfall_log (excluding Energy)
  Communication: N=53
  Construction: N=53
  Electricity: N=53
  Financial Insitutions: N=53
  Manufacturing: N=53
  Mining: N=53
  Rental and other services: N=53
  Social and Administrative: N=53
  Trade: N=53
  Transportation: N=53


alt.LayerChart(...)

### Model 5: QoQ % Change (No Lag)

In [31]:
print("Model 5: gdp_qoq ~ no2_qoq (excluding Energy)")
qoq_no_energy = no2_indicators_qoq.query('economic_activity != "Energy"')
ml_qoq0 = fit_sector_models(qoq_no_energy, "no2_qoq", outcome="gdp_qoq")

star_qoq0 = Stargazer([m for _, m in ml_qoq0])
star_qoq0.custom_columns([a for a, _ in ml_qoq0])
display(star_qoq0)

display(
    plot_coefficients(
        ml_qoq0,
        "no2_qoq",
        title="GDP QoQ ~ NO₂ QoQ (No Lag) (Model 5)",
        subtitle="OLS: gdp_qoq ~ no2_qoq | OMI 2012–2025",
    )
)

Model 5: gdp_qoq ~ no2_qoq (excluding Energy)
  Communication: N=52
  Construction: N=52
  Electricity: N=52
  Financial Insitutions: N=52
  Manufacturing: N=52
  Mining: N=52
  Rental and other services: N=52
  Social and Administrative: N=52
  Trade: N=52
  Transportation: N=52


alt.LayerChart(...)

### Model 6: QoQ % Change (3-Quarter Lag)

In [32]:
print("Model 6: gdp_qoq ~ no2_qoq_lag3 (excluding Energy)")
ml_qoq3 = fit_sector_models(qoq_no_energy, "no2_qoq_lag3", outcome="gdp_qoq")

star_qoq3 = Stargazer([m for _, m in ml_qoq3])
star_qoq3.custom_columns([a for a, _ in ml_qoq3])
display(star_qoq3)

display(
    plot_coefficients(
        ml_qoq3,
        "no2_qoq_lag3",
        title="GDP QoQ ~ NO₂ QoQ (3-Quarter Lag) (Model 6)",
        subtitle="OLS: gdp_qoq ~ no2_qoq_lag3 | OMI 2012–2025",
    )
)

Model 6: gdp_qoq ~ no2_qoq_lag3 (excluding Energy)
  Communication: N=49
  Construction: N=49
  Electricity: N=49
  Financial Insitutions: N=49
  Manufacturing: N=49
  Mining: N=49
  Rental and other services: N=49
  Social and Administrative: N=49
  Trade: N=49
  Transportation: N=49


alt.LayerChart(...)

### Model 7: YoY % Change (No Lag)

In [33]:
print("Model 7: gdp_yoy ~ no2_yoy (excluding Energy)")
yoy_no_energy = no2_indicators_yoy.query('economic_activity != "Energy"')
ml_yoy0 = fit_sector_models(yoy_no_energy, "no2_yoy", outcome="gdp_yoy")

star_yoy0 = Stargazer([m for _, m in ml_yoy0])
star_yoy0.custom_columns([a for a, _ in ml_yoy0])
display(star_yoy0)

display(
    plot_coefficients(
        ml_yoy0,
        "no2_yoy",
        title="GDP YoY ~ NO₂ YoY (No Lag) (Model 7)",
        subtitle="OLS: gdp_yoy ~ no2_yoy | OMI 2012–2025",
    )
)

Model 7: gdp_yoy ~ no2_yoy (excluding Energy)
  Communication: N=49
  Construction: N=49
  Electricity: N=49
  Financial Insitutions: N=49
  Manufacturing: N=49
  Mining: N=49
  Rental and other services: N=49
  Social and Administrative: N=49
  Trade: N=49
  Transportation: N=49


alt.LayerChart(...)

## Cross-Correlation Analysis

Cross-correlation functions (CCF) between NO₂ and GDP at various lags, to identify the lead/lag structure of the relationship.

In [34]:
selected_activities = ["Transportation", "Manufacturing", "Construction", "Trade"]
max_lag = 6

ccf_records = []
for activity in selected_activities:
    act_df = (
        gdp_no2.query("economic_activity == @activity")
        .dropna(subset=["no2", "gdp"])
        .sort_values("date")
    )
    no2_s = act_df["no2"].values
    gdp_s = act_df["gdp"].values
    n = len(no2_s)

    for lag in range(-max_lag, max_lag + 1):
        if lag < 0:
            a = no2_s[:lag]
            b = gdp_s[-lag:]
        elif lag > 0:
            a = no2_s[lag:]
            b = gdp_s[:-lag]
        else:
            a = no2_s
            b = gdp_s
        if len(a) < 5:
            continue
        corr = np.corrcoef(a, b)[0, 1]
        ccf_records.append(
            {
                "economic_activity": activity,
                "lag": lag,
                "correlation": corr,
            }
        )

ccf_df = pd.DataFrame(ccf_records)

alt.Chart(ccf_df).mark_bar().encode(
    x=alt.X("lag:O", title="Lag (quarters, negative = NO₂ leads)"),
    y=alt.Y("correlation:Q", title="Correlation", scale=alt.Scale(domain=[-1, 1])),
    color=alt.condition(
        alt.datum.lag == 0,
        alt.value("firebrick"),
        alt.value("steelblue"),
    ),
    facet=alt.Facet("economic_activity:N", columns=2),
    tooltip=[
        alt.Tooltip("economic_activity:N", title="Sector"),
        alt.Tooltip("lag:O", title="Lag (quarters)"),
        alt.Tooltip("correlation:Q", title="Correlation", format=".3f"),
    ],
).resolve_scale(y="shared").properties(
    width=280,
    height=200,
    title=alt.Title(
        text="Cross-Correlation: NO₂ vs GDP by Sector (National)",
        subtitle="Negative lag = NO₂ leads GDP. Lag 0 = No lag",
    ),
)

alt.Chart(...)

> **Methodological caveat:** These are *associations*, not causal estimates. Omitted variables may confound the relationship. The small sample (~48 quarters per sector) limits statistical power and prevents adding many controls simultaneously.